In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lmt_simulation as sim

plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## Plot 1: Discard rate vs initial excited fraction

Sweep $|c_1|^2$ from 0 to 1. For each value, run 2000 MC trials of `make_atom_states -> do_clearout`.
The empirical discard rate should overlap with the analytical line $1 - |c_1|^2$.

In [ ]:
excited_fractions = np.linspace(0, 1, 21)
n_trials = 2000
discard_rates = []
errors = []

for ef in excited_fractions:
    c1 = np.sqrt(ef)
    c0 = np.sqrt(1 - ef) if ef < 1 else 0.0
    
    n_discards = 0
    for trial in range(n_trials):
        rng = np.random.default_rng(trial)
        m, pos, vel, amp, isg = sim.make_atom_states(c0=c0, c1=c1)
        omega = 2 * np.pi * sim.TRANSITION_FREQUENCY
        squiggly = sim.transform_state_vector(m, amp, isg, omega_laser=omega, t=0, z=0, vz=0, inverse=False)
        result = sim.do_clearout(m, squiggly, isg, pos, vel, rng=rng)
        if result is None:
            n_discards += 1
    
    discard_rates.append(n_discards / n_trials)
    errors.append(4.0 / np.sqrt(n_trials))

plt.figure()
plt.errorbar(excited_fractions, discard_rates, yerr=errors, fmt='o', label='MC empirical')
plt.plot(excited_fractions, 1 - excited_fractions, 'k-', label='Analytical: $1 - |c_1|^2$')
plt.xlabel('Initial excited fraction $|c_1|^2$')
plt.ylabel('Discard rate')
plt.title('Clearout discard rate vs initial excited fraction')
plt.legend()
plt.grid(True)
plt.show()

## Plot 2: Mach-Zehnder with clearout

Standard pi/2 - pi - pi/2 MZ phase scan with `do_clearout` inserted between the pi and final pi/2 pulses.

Expectations:
- P_discarded ~ 0.5 (atoms in ground after the pi pulse are blown away regardless of phi)
- After clearout only excited-state atoms remain; for a resonant pi pulse these are concentrated in a single momentum branch, so the final pi/2 pulse produces no interference fringe — P_ground and P_excited are flat at ~0.25 each (0.5 of the surviving 0.5 population, split equally by the final pi/2).

In [ ]:
T_FREE = 200e-6
DETUNING_HZ = sim.RECOIL_FREQUENCY_HZ
OMEGA_LASER = 2 * np.pi * (sim.TRANSITION_FREQUENCY + DETUNING_HZ)

def mz_sequence_with_clearout(phi, rng):
    """Run a pi/2 - pi - clearout - pi/2 MZ sequence. Returns state tuple or None."""
    m, pos, vel, amp, isg = sim.make_atom_states(c0=1, c1=0)
    current_time = 0.0
    vz = 0.0
    
    squiggly = sim.transform_state_vector(m, amp, isg, omega_laser=OMEGA_LASER, t=current_time, z=0.0, vz=vz, inverse=False)
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI / 2, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=0.0, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI / 2
    
    m, squiggly, isg, pos, vel = sim.propagate_states_in_borde_representation(
        m, squiggly, isg, pos, vel,
        time_of_propegation=T_FREE, omega_laser=OMEGA_LASER, vz=vz, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR,
    )
    current_time += T_FREE
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=phi, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI
    
    result = sim.do_clearout(m, squiggly, isg, pos, vel, rng=rng)
    if result is None:
        return None
    m, squiggly, isg, pos, vel = result
    
    m, squiggly, isg, pos, vel = sim.propagate_states_in_borde_representation(
        m, squiggly, isg, pos, vel,
        time_of_propegation=T_FREE, omega_laser=OMEGA_LASER, vz=vz, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR,
    )
    current_time += T_FREE
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI / 2, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=4 * phi, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI / 2
    
    amp_final = sim.transform_state_vector(m, squiggly, isg, omega_laser=OMEGA_LASER, t=current_time, z=0.0, vz=vz, inverse=True)
    return m, amp_final, isg, pos, vel


phi_values = np.linspace(0, 2 * np.pi, 51)
n_trials = 2000

p_ground = np.empty_like(phi_values)
p_excited = np.empty_like(phi_values)
p_discarded = np.empty_like(phi_values)

for i, phi in enumerate(phi_values):
    pg, pe, pd = sim.run_clearout_trials(
        lambda rng: mz_sequence_with_clearout(phi, rng), n_trials=n_trials
    )
    p_ground[i] = pg
    p_excited[i] = pe
    p_discarded[i] = pd

plt.figure()
plt.plot(phi_values / np.pi, p_ground, 'o-', label='P_ground')
plt.plot(phi_values / np.pi, p_excited, 's-', label='P_excited')
plt.plot(phi_values / np.pi, p_discarded, '^-', label='P_discarded')
plt.xlabel(r'$\phi$ ($\pi$ rad)')
plt.ylabel('Population fraction')
plt.title('Mach-Zehnder with clearout: fringe contrast')
plt.xticks([0, 0.5, 1, 1.5, 2], ['0', 'pi/2', 'pi', '3pi/2', '2pi'])
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 2)
plt.ylim(-0.05, 1.05)
plt.show()

## Plot 3: MC convergence

Pick one phi from Plot 2 (phi = 0.3 rad) and sweep n_trials.
Plot |MC estimate - deterministic baseline| vs n_trials on log-log,
with a 1/sqrt(N) reference line.

In [ ]:
def mz_sequence_deterministic(phi):
    """Deterministic MZ sequence: drop ground rows after pi pulse, renormalise excited state."""
    m, pos, vel, amp, isg = sim.make_atom_states(c0=1, c1=0)
    current_time = 0.0
    vz = 0.0
    
    squiggly = sim.transform_state_vector(m, amp, isg, omega_laser=OMEGA_LASER, t=current_time, z=0.0, vz=vz, inverse=False)
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI / 2, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=0.0, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI / 2
    
    m, squiggly, isg, pos, vel = sim.propagate_states_in_borde_representation(
        m, squiggly, isg, pos, vel,
        time_of_propegation=T_FREE, omega_laser=OMEGA_LASER, vz=vz, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR,
    )
    current_time += T_FREE
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=phi, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI
    
    p_g, p_e = sim.calculate_ground_and_excited_probabilities(m, squiggly, isg)
    total = p_g + p_e
    p_discard = p_g / total
    p_survive = p_e / total
    
    # Renormalise excited rows (matching do_clearout behaviour)
    keep = ~isg
    m = m[keep]
    squiggly = squiggly[keep] * (1.0 / np.sqrt(p_e))
    isg = isg[keep]
    pos = pos[keep]
    vel = vel[keep]
    
    m, squiggly, isg, pos, vel = sim.propagate_states_in_borde_representation(
        m, squiggly, isg, pos, vel,
        time_of_propegation=T_FREE, omega_laser=OMEGA_LASER, vz=vz, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR,
    )
    current_time += T_FREE
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI / 2, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=4 * phi, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI / 2
    
    amp_final = sim.transform_state_vector(m, squiggly, isg, omega_laser=OMEGA_LASER, t=current_time, z=0.0, vz=vz, inverse=True)
    g, e = sim.calculate_ground_and_excited_probabilities(m, amp_final, isg)
    
    # Weight by survival probability
    return p_survive * g, p_survive * e, p_discard


phi_test = 0.3
baseline_g, baseline_e, baseline_d = mz_sequence_deterministic(phi_test)
baseline = np.array([baseline_g, baseline_e, baseline_d])

n_trials_list = [50, 200, 1000, 5000, 20000]
l2_errors = []

for n in n_trials_list:
    pg, pe, pd = sim.run_clearout_trials(
        lambda rng: mz_sequence_with_clearout(phi_test, rng), n_trials=n
    )
    mc = np.array([pg, pe, pd])
    l2_errors.append(np.linalg.norm(mc - baseline))

ref = np.array(n_trials_list, dtype=float)
ref_line = l2_errors[0] * np.sqrt(ref[0]) / np.sqrt(ref)

plt.figure()
plt.loglog(n_trials_list, l2_errors, 'o-', label='|MC - baseline|_2')
plt.loglog(n_trials_list, ref_line, 'k--', label='propto 1/sqrt(N)')
plt.xlabel('Number of trials N')
plt.ylabel('L2 error')
plt.title(f'MC convergence at phi = {phi_test} rad')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.show()

## Plot 4: Clearout commutes with free propagation

Compare do_clearout immediately followed by free propagation vs free propagation followed by do_clearout.
Both variants use the same RNG seed and free-fall duration.
The result distributions should overlay within MC noise, demonstrating the documented frame/time independence of the projection.

In [ ]:
def mz_variant_a(phi, rng):
    """Variant A: clearout immediately after pi pulse, then propagate."""
    m, pos, vel, amp, isg = sim.make_atom_states(c0=1, c1=0)
    current_time = 0.0
    vz = 0.0
    
    squiggly = sim.transform_state_vector(m, amp, isg, omega_laser=OMEGA_LASER, t=current_time, z=0.0, vz=vz, inverse=False)
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI / 2, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=0.0, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI / 2
    
    m, squiggly, isg, pos, vel = sim.propagate_states_in_borde_representation(
        m, squiggly, isg, pos, vel,
        time_of_propegation=T_FREE, omega_laser=OMEGA_LASER, vz=vz, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR,
    )
    current_time += T_FREE
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=phi, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI
    
    result = sim.do_clearout(m, squiggly, isg, pos, vel, rng=rng)
    if result is None:
        return None
    m, squiggly, isg, pos, vel = result
    
    m, squiggly, isg, pos, vel = sim.propagate_states_in_borde_representation(
        m, squiggly, isg, pos, vel,
        time_of_propegation=T_FREE, omega_laser=OMEGA_LASER, vz=vz, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR,
    )
    current_time += T_FREE
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI / 2, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=4 * phi, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI / 2
    
    amp_final = sim.transform_state_vector(m, squiggly, isg, omega_laser=OMEGA_LASER, t=current_time, z=0.0, vz=vz, inverse=True)
    return m, amp_final, isg, pos, vel


def mz_variant_b(phi, rng):
    """Variant B: propagate first, then clearout."""
    m, pos, vel, amp, isg = sim.make_atom_states(c0=1, c1=0)
    current_time = 0.0
    vz = 0.0
    
    squiggly = sim.transform_state_vector(m, amp, isg, omega_laser=OMEGA_LASER, t=current_time, z=0.0, vz=vz, inverse=False)
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI / 2, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=0.0, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI / 2
    
    m, squiggly, isg, pos, vel = sim.propagate_states_in_borde_representation(
        m, squiggly, isg, pos, vel,
        time_of_propegation=T_FREE, omega_laser=OMEGA_LASER, vz=vz, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR,
    )
    current_time += T_FREE
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=phi, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI
    
    m, squiggly, isg, pos, vel = sim.propagate_states_in_borde_representation(
        m, squiggly, isg, pos, vel,
        time_of_propegation=T_FREE, omega_laser=OMEGA_LASER, vz=vz, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR,
    )
    current_time += T_FREE
    
    result = sim.do_clearout(m, squiggly, isg, pos, vel, rng=rng)
    if result is None:
        return None
    m, squiggly, isg, pos, vel = result
    
    m, squiggly, isg, pos, vel = sim.pulse_interaction_in_borde_representation(
        m, squiggly, isg, pos, vel,
        pulse_detuning=DETUNING_HZ, t_pulse=sim.T_PI / 2, pulse_rabi_freq=sim.RABI_FREQ,
        pulse_phase=4 * phi, k_sign=+1, k_wavevector=sim.K_WAVEVECTOR, vz=vz,
    )
    current_time += sim.T_PI / 2
    
    amp_final = sim.transform_state_vector(m, squiggly, isg, omega_laser=OMEGA_LASER, t=current_time, z=0.0, vz=vz, inverse=True)
    return m, amp_final, isg, pos, vel


phi_values_commute = np.linspace(0, 2 * np.pi, 21)
n_trials_commute = 2000

p_g_a = np.empty_like(phi_values_commute)
p_e_a = np.empty_like(phi_values_commute)
p_d_a = np.empty_like(phi_values_commute)
p_g_b = np.empty_like(phi_values_commute)
p_e_b = np.empty_like(phi_values_commute)
p_d_b = np.empty_like(phi_values_commute)

for i, phi in enumerate(phi_values_commute):
    base_seed = 12345 + i
    rng_a = np.random.default_rng(base_seed)
    rng_b = np.random.default_rng(base_seed)
    
    p_g_a[i], p_e_a[i], p_d_a[i] = sim.run_clearout_trials(
        lambda rng: mz_variant_a(phi, rng), n_trials=n_trials_commute, rng=rng_a
    )
    p_g_b[i], p_e_b[i], p_d_b[i] = sim.run_clearout_trials(
        lambda rng: mz_variant_b(phi, rng), n_trials=n_trials_commute, rng=rng_b
    )

plt.figure()
plt.plot(phi_values_commute / np.pi, p_g_a, 'o-', color='tab:blue', label='Variant A: P_ground')
plt.plot(phi_values_commute / np.pi, p_e_a, 's-', color='tab:orange', label='Variant A: P_excited')
plt.plot(phi_values_commute / np.pi, p_d_a, '^-', color='tab:green', label='Variant A: P_discarded')
plt.plot(phi_values_commute / np.pi, p_g_b, 'o--', color='tab:blue', alpha=0.6, label='Variant B: P_ground')
plt.plot(phi_values_commute / np.pi, p_e_b, 's--', color='tab:orange', alpha=0.6, label='Variant B: P_excited')
plt.plot(phi_values_commute / np.pi, p_d_b, '^--', color='tab:green', alpha=0.6, label='Variant B: P_discarded')
plt.xlabel(r'$\phi$ ($\pi$ rad)')
plt.ylabel('Population fraction')
plt.title('Clearout commutes with free propagation')
plt.xticks([0, 0.5, 1, 1.5, 2], ['0', 'pi/2', 'pi', '3pi/2', '2pi'])
plt.legend(ncol=2, fontsize=8)
plt.grid(True, alpha=0.3)
plt.xlim(0, 2)
plt.ylim(-0.05, 1.05)
plt.show()